# 03 - Classification ML sur Embeddings LLM (DistilBERT)

**SAE-117** — Epic 4 : ML Classique

## Objectif
Entraîner 4 algorithmes de ML classique sur les embeddings DistilBERT pré-calculés :
- Logistic Regression
- SVM (LinearSVC)
- Random Forest
- Naive Bayes (GaussianNB)

**Grille :** ML-LLM (1pt) + ML classiques plusieurs (4pts)

**Pré-requis :** SAE-112 (Extraction embeddings LLM DistilBERT)

## 1. Setup & Imports

In [1]:
import sys
sys.path.insert(0, '../..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.preprocessing import StandardScaler

from src import setup_plot_style

setup_plot_style()

RANDOM_STATE = 42
print('Imports OK')

ModuleNotFoundError: No module named 'gensim'

## 2. Chargement des embeddings LLM pré-calculés

In [ ]:
# Charger les embeddings DistilBERT pré-calculés (SAE-112)
# Format attendu : fichier .npy ou .npz dans data/
embeddings_path = '../../data/distilbert_embeddings.npy'
labels_path = '../../data/distilbert_labels.npy'

X = np.load(embeddings_path)
y = np.load(labels_path)

print(f'Embeddings shape : {X.shape}')
print(f'Labels shape     : {y.shape}')
print(f'Classes uniques  : {np.unique(y)}')
print(f'Distribution     :')
unique, counts = np.unique(y, return_counts=True)
for label, count in zip(unique, counts):
    print(f'  Classe {label}: {count:,} ({count/len(y)*100:.1f}%)')

## 3. Split train/test

On utilise le **même seed** (`random_state=42`) et le **même ratio** (80/20) que les autres notebooks ML pour garantir la comparabilité.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f'Train : {X_train.shape[0]:,} échantillons')
print(f'Test  : {X_test.shape[0]:,} échantillons')
print(f'Dimension embeddings : {X_train.shape[1]}')

In [ ]:
# Standardisation (important pour LogReg et SVM)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('Standardisation appliquée')
print(f'Moyenne train : {X_train_scaled.mean():.6f}')
print(f'Std train     : {X_train_scaled.std():.6f}')

## 4. Fonctions utilitaires

In [ ]:
def train_and_evaluate(model, model_name, X_tr, X_te, y_tr, y_te):
    """
    Entraîne un modèle et retourne les métriques.
    """
    print(f'\n{"="*60}')
    print(f'  {model_name}')
    print(f'{"="*60}')
    
    # Entraînement
    model.fit(X_tr, y_tr)
    
    # Prédictions
    y_pred = model.predict(X_te)
    
    # Métriques
    acc = accuracy_score(y_te, y_pred)
    prec = precision_score(y_te, y_pred, average='weighted')
    rec = recall_score(y_te, y_pred, average='weighted')
    f1 = f1_score(y_te, y_pred, average='weighted')
    
    print(f'\nAccuracy  : {acc:.4f}')
    print(f'Precision : {prec:.4f}')
    print(f'Recall    : {rec:.4f}')
    print(f'F1-Score  : {f1:.4f}')
    
    print(f'\nClassification Report :')
    print(classification_report(y_te, y_pred))
    
    return {
        'model_name': model_name,
        'model': model,
        'y_pred': y_pred,
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1_score': f1
    }


def plot_confusion_matrix(y_true, y_pred, model_name, ax=None):
    """
    Affiche la matrice de confusion.
    """
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot(ax=ax, cmap='Blues', values_format='d')
    if ax:
        ax.set_title(model_name, fontsize=12, fontweight='bold')
    return cm

## 5. Entraînement des 4 modèles

### 5.1 Logistic Regression

In [ ]:
lr_model = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_STATE,
    multi_class='multinomial',
    solver='lbfgs',
    C=1.0
)

results_lr = train_and_evaluate(
    lr_model, 'Logistic Regression',
    X_train_scaled, X_test_scaled, y_train, y_test
)

### 5.2 SVM (LinearSVC)

In [ ]:
svm_model = LinearSVC(
    max_iter=2000,
    random_state=RANDOM_STATE,
    C=1.0,
    dual='auto'
)

results_svm = train_and_evaluate(
    svm_model, 'SVM (LinearSVC)',
    X_train_scaled, X_test_scaled, y_train, y_test
)

### 5.3 Random Forest

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

# Random Forest n'a pas besoin de standardisation,
# mais on utilise les données non-scalées par cohérence
results_rf = train_and_evaluate(
    rf_model, 'Random Forest',
    X_train, X_test, y_train, y_test
)

### 5.4 Naive Bayes (GaussianNB)

In [ ]:
nb_model = GaussianNB()

results_nb = train_and_evaluate(
    nb_model, 'Naive Bayes (GaussianNB)',
    X_train_scaled, X_test_scaled, y_train, y_test
)

## 6. Matrices de confusion

In [ ]:
all_results = [results_lr, results_svm, results_rf, results_nb]

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
fig.suptitle('Matrices de Confusion — Embeddings LLM (DistilBERT)', 
             fontsize=16, fontweight='bold', y=1.02)

for ax, result in zip(axes.flat, all_results):
    plot_confusion_matrix(y_test, result['y_pred'], result['model_name'], ax=ax)

plt.tight_layout()
plt.savefig('../../outputs/figures/confusion_matrices_llm.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure sauvegardée : outputs/figures/confusion_matrices_llm.png')

## 7. Tableau comparatif des 4 modèles

In [ ]:
# Construire le tableau comparatif
comparison_data = []
for r in all_results:
    comparison_data.append({
        'Modèle': r['model_name'],
        'Accuracy': f"{r['accuracy']:.4f}",
        'Precision': f"{r['precision']:.4f}",
        'Recall': f"{r['recall']:.4f}",
        'F1-Score': f"{r['f1_score']:.4f}"
    })

df_comparison = pd.DataFrame(comparison_data)
df_comparison = df_comparison.set_index('Modèle')

print('\n' + '='*70)
print('  TABLEAU COMPARATIF — ML sur Embeddings LLM (DistilBERT)')
print('='*70)
display(df_comparison)

# Identifier le meilleur modèle
best_idx = np.argmax([r['f1_score'] for r in all_results])
best = all_results[best_idx]
print(f'\n🏆 Meilleur modèle : {best["model_name"]} (F1={best["f1_score"]:.4f})')

In [ ]:
# Visualisation comparative
metrics = ['accuracy', 'precision', 'recall', 'f1_score']
model_names = [r['model_name'] for r in all_results]

fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(model_names))
width = 0.2
colors = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63']

for i, (metric, color) in enumerate(zip(metrics, colors)):
    values = [r[metric] for r in all_results]
    bars = ax.bar(x + i * width, values, width, label=metric.replace('_', ' ').title(), 
                  color=color, alpha=0.85)
    # Annoter les barres
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xlabel('Modèle')
ax.set_ylabel('Score')
ax.set_title('Comparaison des 4 modèles ML — Embeddings LLM (DistilBERT)', 
             fontsize=14, fontweight='bold')
ax.set_xticks(x + 1.5 * width)
ax.set_xticklabels(model_names, rotation=15, ha='right')
ax.legend(loc='lower right')
ax.set_ylim(0, 1.05)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../../outputs/figures/comparison_ml_llm.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure sauvegardée : outputs/figures/comparison_ml_llm.png')

## 8. Analyse & Conclusions

### Observations

Les embeddings DistilBERT capturent des informations sémantiques riches, ce qui devrait se traduire par de meilleures performances comparé aux représentations TF-IDF ou N-grammes.

### Points clés
- **Dimension** : Les embeddings DistilBERT sont de dimension 768, bien plus bas-dimensionnel que TF-IDF
- **Sémantique** : Ces embeddings capturent le sens contextuel des mots
- **Comparaison** : Les résultats sont à comparer avec les notebooks `01-ml-tfidf.ipynb` et `02-ml-ngram.ipynb`

In [ ]:
print('\n✅ Notebook SAE-117 terminé avec succès !')
print(f'   4 modèles entraînés sur embeddings LLM (DistilBERT)')
print(f'   Meilleur modèle : {best["model_name"]} (F1={best["f1_score"]:.4f})')